# Module 04.01 — Data Views (`index-pattern`)

> Part of the **Elasticsearch Snapshot Training** course.
> Run `00_setup.ipynb` first to start the Docker stack and load sample data.


In [ ]:
import sys, json, time
sys.path.insert(0, '..')
from helpers import *

es = get_client()

---
## 4.1 Data Views (`index-pattern`)

**Data Views** (formerly called Index Patterns) are the foundation of Kibana's data layer.
Every time Kibana queries Elasticsearch — to draw a chart, populate Discover, or run an
alert — it does so through a Data View. The Data View tells Kibana which index pattern to
target (wildcards allowed, e.g. `logs-*`), which field holds the timestamp, and how each
field should be displayed or formatted.

Because virtually every other saved object ultimately references a Data View (searches,
visualizations, Lens charts, maps, dashboards, alerts all have at least one
`index-pattern` in their `references` array), **Data Views are the single most important
type to get right when restoring**. If a Data View is missing after a restore the objects
that reference it will silently break — charts will refuse to load and Discover will show
an error. Kibana's restore ordering ensures Data Views are always created before anything
that depends on them.

Key attributes stored in the saved object:
- `title` — the index pattern glob (e.g. `kibana_sample_data_ecommerce`)
- `timeFieldName` — the default time field used by all time-series charts
- `fieldFormatMap` — per-field display overrides (e.g. currency symbol for a price field)
- `runtimeFieldMap` — Painless-based runtime fields defined at the Kibana layer

### Create in Kibana UI
1. Go to **Stack Management → Kibana → Data Views**
2. Click **Create data view**
3. Enter pattern `training-orders-*`, name it `Training Orders`
4. Select `@timestamp` as the time field
5. Click **Save data view to Kibana**

We will also create one via the API for the exercise:

In [ ]:
heading('4.1 Data View — create')

# Idempotent: reuse existing "Training Orders" data view if it already exists
existing = find_saved_objects('index-pattern')
existing_dv = next((o for o in existing if o['attributes'].get('name') == 'Training Orders'), None)
if existing_dv:
    dv_id = existing_dv['id']
    info(f'Data view already exists: id={dv_id}')
else:
    dv_response = kibana_post('/api/data_views/data_view', {
        'data_view': {
            'title': 'training-orders-*',
            'name': 'Training Orders',
            'timeFieldName': '@timestamp',
        },
    })
    dv_id = dv_response['data_view']['id']
    success(f'Data view created: id={dv_id}')

In [ ]:
# ── Kibana UI ────────────────────────────────────────────────────────────
# The data view we just created should appear here:
kibana_link('/app/management/kibana/dataViews', 'Stack Management → Kibana → Data Views')

In [ ]:
# Inspect the saved object schema
heading('Data View saved object schema')
obj_list = find_saved_objects('index-pattern')
training_dv = next((o for o in obj_list if o['attributes'].get('name') == 'Training Orders'), None)
if training_dv:
    pp(training_dv, 'index-pattern saved object')
    console.print('')
    info('Key fields in attributes:')
    info('  title           — the index pattern string (wildcards allowed)')
    info('  timeFieldName   — default time field')
    info('  fields          — JSON string: field type overrides')
    info('  fieldFormatMap  — JSON string: display format per field')
    info('  runtimeFieldMap — runtime fields defined in this data view')

In [ ]:
# Ensure the data view exists before snapshotting (makes this cell re-runnable)
existing = find_saved_objects('index-pattern')
if not any(o['id'] == dv_id for o in existing):
    warn(f'Data view {dv_id} missing — recreating before snapshot')
    dv_response = kibana_post('/api/data_views/data_view', {
        'data_view': {
            'title': 'training-orders-*',
            'name': 'Training Orders',
            'timeFieldName': '@timestamp',
        },
    })
    dv_id = dv_response['data_view']['id']
    success(f'Recreated data view: id={dv_id}')

# Snapshot → delete → restore
snap_delete_restore_cycle('snap-dv-test', 'Data View')

# Delete it
kibana_delete(f'/api/data_views/data_view/{dv_id}')
warn(f'Deleted data view {dv_id}')
assert not any(o['id'] == dv_id for o in find_saved_objects('index-pattern')), 'Should be gone'

# Restore
restore_kibana_state(es, SNAP_REPO, 'snap-dv-test')
time.sleep(3)  # allow Kibana to process restored objects

# Confirm
restored = find_saved_objects('index-pattern')
assert any(o['id'] == dv_id for o in restored), 'Data view should be restored'
success(f'Data view {dv_id} successfully restored!')